In [ ]:
from pathlib import Path
from joblib import Memory
from google.cloud import storage

from vacant_lot.config import load_config
from vacant_lot.gee_utils import (
    init_gee,
    load_image_collection,
    # Vector workflow functions
    upload_parcels_to_gcs,
    ingest_parcels_to_gee,
    reduce_regions_to_gcs,
    # Raster workflow functions (for reference)
    upload_raster_to_gcs,
    ingest_raster_to_gee,
)
from vacant_lot.data_utils import upload_to_gcs
from vacant_lot.analysis import cluster_spectral_data
from vacant_lot.mappluto import load_and_sample
from vacant_lot.geometry import get_city_geometry
from vacant_lot.naip import calculate_spectral_indices
from vacant_lot.raster_utils import rasterize_parcels
from vacant_lot.analysis import (
    read_csv_from_gcs,
    analyze_cluster_composition,
    cluster_dataframe
)
from vacant_lot.plotting import plot_pca, plot_cluster_composition, plot_feature_importance, plot_pca_loadings

In [ ]:
# ===== RUN CONFIGURATION =====
CONFIG_FILE = "nyc_buildings.yaml"  # Change to "nyc_buildings", etc.
# =============================

memory = Memory(Path("cache"), verbose=0)

# Load run-specific config
CONFIG = load_config(CONFIG_FILE)

# Create output directories
CONFIG.ensure_output_dirs()

# gcs client
client = storage.Client()
bucket = client.bucket(CONFIG.gcp.bucket)

In [ ]:
init_gee(CONFIG.gcp)

In [ ]:
"""
- Load MapPLUTO (NYC) 2022 v3 -- fall matches with NAIP
- Reproject to output_crs (EPSG:32618) for geometry-derived metric area computation
- Filter by area: min from min_pixels × resolution²
- Sample vacant lots based on land use codes (configurable)
- Returns GeoDataFrames in EPSG:32618 (projected CRS)
- NOTE: If GEE uploads are re-enabled, data must be converted to EPSG:4326 first
"""
mappluto_22_gdf, sampled_gdf = load_and_sample(
    path=CONFIG.get_parcel_path(),
    layer=CONFIG.parcel.layer,
    land_use_codes=CONFIG.parcel.vacant_codes,
    col_to_sample=CONFIG.parcel.landuse_column,
    projected_crs=CONFIG.raster.output_crs,
    vacant_min_fraction=CONFIG.sampling.vacant_min_fraction,
    total_samples=CONFIG.sampling.total_samples,
    resolution=CONFIG.raster.resolution,
    min_pixels=CONFIG.sampling.min_pixels,
    random_state=CONFIG.sampling.random_state,
)

In [ ]:
city_geom = get_city_geometry(CONFIG)

@memory.cache
def get_naip_for_city(config):
    return load_image_collection(
        collection_id=config.naip.collection_id,
        start_date=config.naip.start_date,
        end_date=config.naip.end_date,
        region=city_geom,
        mosaic=True
    )

naip = get_naip_for_city(CONFIG)
naip = calculate_spectral_indices(naip, CONFIG)

# Vector Based Reduction

## Parcel Table Ingestion to GEE
- run when parcels should be overwritten in both gcs and gee 

In [ ]:
# gcs_uri = upload_parcels_to_gcs(
#     gdf=sampled_gdf,
#     id_column=CONFIG.parcel.id_column,
#     output_dir=CONFIG.get_intermediaries_dir(),
#     bucket_name=CONFIG.gcp.bucket,
#     gcs_prefix=CONFIG.gee.export_prefix,
#     filename_prefix=f"parcels",
# )

In [ ]:
# ingest_parcels_to_gee(
#     asset_id=f"projects/{CONFIG.gcp.project_id}/assets/{CONFIG.city}_{CONFIG.run_name}_parcels",
#     gcs_uri=gcs_uri,
# )

"""
Can provide bucket_name, gcs_prefix, gcs_filename if cell above doesn't need to be re run 
f"gs://{bucket_name}/{gcs_prefix}/{gcs_filename}"
"""
# ingest_parcels_to_gee(
#     asset_id=f"projects/{CONFIG.gcp.project_id}/assets/{CONFIG.city}_parcels",
#     bucket_name=CONFIG.gcp.bucket,
#     gcs_prefix=CONFIG.gee.export_prefix,
#     gcs_filename=f"{CONFIG.city}_parcels.zip",
# )

## Reduce Pixel Values Per Parcel
- only run once after asset exists in gee --> uploads a csv of parcel_identifier, avg_spectral_stat_1, avg_spectral_stat_2, ..., avg_spectral_stat_n to gcs

In [ ]:
# task = reduce_regions_to_gcs(
#     imagery=naip,
#     parcels_asset_id=f"projects/{CONFIG.gcp.project_id}/assets/{CONFIG.city}_{CONFIG.run_name}_parcels",
#     bucket_name=CONFIG.gcp.bucket,
#     gcs_prefix=CONFIG.gee.export_prefix,
#     include_count=True,
# )

## Start here to get the Parcel by Parcel Spectral Stats

In [ ]:
spectral_df = read_csv_from_gcs(CONFIG.gcp.bucket, f"{CONFIG.gee.export_prefix}/parcel_spectral_stats.csv")
spectral_df

In [ ]:
mappluto_features = sampled_gdf[["BBL", f"{CONFIG.parcel.landuse_column}", "geometry", "geom_perimeter_epsg32618", "area_m2_epsg32618"]].copy()
mappluto_features

In [ ]:
mappluto_features

In [ ]:
# spectral only clustering 
mappluto_spectral_df = mappluto_features[["BBL", f"{CONFIG.parcel.landuse_column}"]].copy()

#join on bbl with spectral df 
merged_df = spectral_df.merge(mappluto_spectral_df, on="BBL", how="left")
merged_df

In [ ]:
# Kmeans + PCA clustering

merged_df, models = cluster_dataframe(
    df=merged_df,
    feature_columns=CONFIG.clustering.features if CONFIG.clustering else None,
    n_clusters=CONFIG.clustering.n_clusters if CONFIG.clustering else 5,
    random_state=CONFIG.clustering.random_state if CONFIG.clustering else 42,
    add_pca=True,
)
merged_df

In [ ]:
plot_pca(                                                                                                                                      
    df=merged_df,                                                                                                                              
    output_dir=CONFIG.get_figures_dir(),
    color_col="cluster",
    highlight_col=CONFIG.parcel.landuse_column,
    highlight_value=CONFIG.parcel.vacant_codes,   # vacant lots drawn on top in yellow
    title="PCA — NYC Parcels by Cluster (vacant lots highlighted)",
)

In [ ]:
out_path, composition_df = plot_cluster_composition(merged_df, output_dir=CONFIG.get_figures_dir(), category_col=CONFIG.parcel.landuse_column, vacant_codes=CONFIG.parcel.vacant_codes)

In [ ]:
plot_pca_loadings(models, output_dir=CONFIG.get_figures_dir())

In [ ]:
plot_feature_importance(models, output_dir=CONFIG.get_figures_dir())

# Rasterize Parcels

In [ ]:
# raster_output_path = CONFIG.get_raster_path(REPO_ROOT)
# intermediaries_dir = CONFIG.get_intermediaries_dir(REPO_ROOT)
# mapping_output_path = intermediaries_dir / "parcel_id_mapping.csv"

# raster_path, mapping_path = rasterize_parcels(
#     gdf=mappluto_22_gdf,
#     id_column=CONFIG.parcel.id_column,
#     raster_output_path=raster_output_path,
#     mapping_output_path=mapping_output_path,
#     crs=CONFIG.raster.output_crs,
#     resolution=CONFIG.raster.resolution,
#     overwrite=False,
# )

The following cell should only be run once when the raster needs to be uploaded to gcs and then ingested to gee

In [ ]:
# ## Upload parcel raster to GEE

# # Upload to GCS first
# gcs_uri = upload_raster_to_gcs(
#     raster_path=raster_path,
#     gcs_bucket=CONFIG.gcp.bucket,
#     gcs_prefix=CONFIG.gee.export_prefix,
#     gcs_filename="parcels_raster.tif",
# )

# # Ingest to GEE asset
# task_info = ingest_raster_to_gee(
#     gcs_uri=gcs_uri,
#     asset_id=CONFIG.gee.parcel_asset,
# )

# # Task runs asynchronously - check progress in Earth Engine Code Editor
# print(f"Task {task_info['task_id']} submitted")
# print(f"Check progress at: https://code.earthengine.google.com/tasks")

## Raster-Based Parcel Reduction

Instead of using `reduceRegions()` with vector polygons (which is slow for many parcels), we can use a grouped reducer with the parcel raster. This approach:
1. Uploads the parcel raster to GEE as an Image asset
2. Stacks the imagery bands with the parcel ID band
3. Uses `reduceRegion()` with a grouped reducer to compute stats per parcel ID

Only upload to GEE when raster asset for a specific city needs to be overwritten. This is a time consuming process.

In [ ]:
# # Raster-based grouped reducer with ONLY sampled parcels
# from vacant_lot.gee_utils import load_parcel_raster_asset, reduce_by_parcel_raster, export_grouped_stats_to_gcs
# import pandas as pd

# # Load parcel raster from GEE
# parcel_raster_asset = CONFIG.gee.parcel_asset
# parcel_raster = load_parcel_raster_asset(parcel_raster_asset)

# # Load the parcel_id mapping (BBL -> rasterized integer ID)
# mapping_path = CONFIG.get_intermediaries_dir(REPO_ROOT) / "parcel_id_mapping.csv"
# parcel_mapping = pd.read_csv(mapping_path)
# print(f"Loaded parcel mapping: {len(parcel_mapping)} parcels")

# # Get the rasterized IDs for sampled parcels only
# # Join sampled_gdf with mapping on BBL
# sampled_bbls = sampled_gdf['BBL'].astype(int).tolist()
# sampled_mapping = parcel_mapping[parcel_mapping['BBL'].isin(sampled_bbls)]
# sampled_parcel_ids = sampled_mapping['parcel_id'].tolist()
# print(f"Sampled parcels with mapping: {len(sampled_parcel_ids)}")

# # Verify bands
# print("NAIP bands:", naip.bandNames().getInfo())
# bands_to_reduce = ['R', 'G', 'B', 'N', 'NDVI', 'SAVI', 'Brightness', 'BareSoilProxy']

# naip_32618 = naip.reproject("EPSG:32618")
# # Run reduction with ONLY sampled parcel IDs
# grouped_stats = reduce_by_parcel_raster(
#     imagery=naip_32618,
#     parcel_raster=parcel_raster,
#     region=city_geom,
#     bands=bands_to_reduce,
#     scale=1,
#     debug=True,
#     parcel_ids=sampled_parcel_ids,  # Only process sampled parcels!
# )

# # Export results to GCS with new path structure
# task = export_grouped_stats_to_gcs(
#     stats_dict=grouped_stats,
#     description='parcel_spectral_stats_raster',
#     bucket=CONFIG.gcp.bucket,
#     file_prefix=f'{CONFIG.gee.export_prefix}/parcel_spectral_stats_raster',  # New path: eda/new_york_new_york/
# )

In [ ]:
# Generate README for this run
from vacant_lot.config import generate_run_readme

run_stats = {
    "Total parcels loaded": len(mappluto_22_gdf),
    "Sampled parcels": len(sampled_gdf),
    "Land use codes": sorted(sampled_gdf[CONFIG.parcel.landuse_column].unique()),
    "Unique clusters": len(merged_df['cluster'].unique()),
    "Total spectral features": len(CONFIG.clustering.features),
}

generate_run_readme(
    config=CONFIG,
    output_dir=CONFIG.get_output_dir(),
    stats=run_stats,
)